# 01 — Data Preprocessing, Wrangling & EDA

**BSM Mineral Image Classification** · reproduction of *D-ResNet: deep residual neural network for exploration, identification and classification of beach sand minerals*

This notebook performs every data-processing stage described in the paper's
methodology: dataset discovery, inventory, validation, duplicate detection,
data wrangling, metadata imputation, EDA, augmentation, and the
source-group-level split.

### Order of operations — deliberate deviation
The paper augments first and splits afterwards, which places rotated/flipped
siblings of a training image into the test set. This notebook **splits first at
the source-image group level, then augments the training split only**, as
required by the project specification's data-leakage prevention rule.

### Outputs
All artefacts are written under `results/`.

In [1]:
# ------------------------------------------------------------------
# 1. Imports / setup
# ------------------------------------------------------------------
import os
import sys
import json
import random
import hashlib
import platform
import warnings
import datetime
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# ------------------------------------------------------------------
# Global configuration
# ------------------------------------------------------------------
SEED = 42
# Class order follows the paper's presentation order and is fixed for the
# whole project. Every checkpoint and every CSV uses this exact index order.
CLASSES = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

IMG_SIZE = 224
SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Paper-reported values, kept ONLY as reference targets for reporting.
PAPER_REPORTED_IMAGES = 357
PAPER_REPORTED_AUGMENTED = 1468
PAPER_CLASS_COUNTS = {
    "Ilmenite": 60, "Rutile": 62, "Zircon": 60,
    "Garnet": 66, "Monazite": 60, "Sillimanite": 59,
}
PAPER_SOURCE = "Kaggle (per the paper)"

random.seed(SEED)
np.random.seed(SEED)

def find_project_root(start=None):
    # Notebooks are run from inside notebooks/ on Colab and from the repo
    # root locally, so resolve the root instead of hard-coding any path.
    p = Path(start or Path.cwd()).resolve()
    for cand in [p] + list(p.parents):
        if (cand / "notebooks").is_dir() and (cand / "results").is_dir():
            return cand
    return p

PROJECT_ROOT = find_project_root()
RESULTS = PROJECT_ROOT / "results"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROC = PROJECT_ROOT / "data" / "processed"
DATA_AUG = PROJECT_ROOT / "data" / "augmented"

for sub in ["dataset_summary", "preprocessing", "augmentation", "metrics",
            "confusion_matrices", "training_curves", "comparisons"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)

def R(sub):
    return RESULTS / sub

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def save_csv(df, subdir, name):
    p = R(subdir) / name
    df.to_csv(p, index=False)
    print(f"  wrote {p.relative_to(PROJECT_ROOT)}  ({len(df)} rows)")
    return p

def banner(n, title):
    print("\n" + "=" * 72)
    print(f"  STEP {n}. {title}")
    print("=" * 72)

banner(1, "Imports / setup")
print("python           :", platform.python_version())
print("PROJECT_ROOT     :", PROJECT_ROOT)
print("RESULTS          :", RESULTS)
print("classes          :", CLASSES)
print("image size       :", IMG_SIZE)
print("seed             :", SEED)
print("in google colab  :", "google.colab" in sys.modules)


  STEP 1. Imports / setup
python           : 3.13.15
PROJECT_ROOT     : /content/BSM-Project
RESULTS          : /content/BSM-Project/results
classes          : ['Ilmenite', 'Rutile', 'Zircon', 'Garnet', 'Monazite', 'Sillimanite']
image size       : 224
seed             : 42
in google colab  : True


---
## 2. Dataset discovery

Locates the six class folders. Order of preference:

1. an already-present local/Colab copy under `data/raw/BSM/`
2. a public Kaggle download (`prasannavenkatesant/beach-sand-mineral-bsm`)
3. otherwise **stop** — the dataset is never fabricated or substituted

**Credentials are never hard-coded.** They are read from Colab secrets, or from
an uploaded `/content/kaggle.json`. Nothing secret is written into this notebook.

In [2]:
KAGGLE_DATASET = "prasannavenkatesant/beach-sand-mineral-bsm"


def find_class_root():
    # A class root is the directory whose children are the six class folders.
    candidates = []
    for base in [DATA_RAW, Path("/content"), Path("/content/drive/MyDrive")]:
        if not base.exists():
            continue
        for cand in [base] + [p for p in base.glob("**/BSM") if p.is_dir()][:50]:
            if not cand.is_dir():
                continue
            kids = {d.name.strip() for d in cand.iterdir() if d.is_dir()}
            if len(kids & set(CLASSES)) >= 4:
                candidates.append(cand)
    return candidates[0] if candidates else None


def kaggle_credentials_available():
    # Order matters. Under `nbconvert --execute` the notebook runs in a
    # subprocess, and Colab secrets are readable ONLY from the interactive
    # Colab UI kernel - userdata.get() raises TimeoutException in a subprocess
    # and blocks for about a minute. So check on-disk / environment credentials
    # FIRST and treat Colab secrets as a last resort.
    local = Path("/content/kaggle.json")
    if local.exists():
        try:
            cred = json.loads(local.read_text())
            if cred.get("username") and cred.get("key"):
                os.environ["KAGGLE_USERNAME"] = cred["username"]
                os.environ["KAGGLE_KEY"] = cred["key"]
                return "/content/kaggle.json"
        except Exception as e:
            print("  debug: /content/kaggle.json unreadable:", type(e).__name__, e)

    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return "environment"

    try:
        from google.colab import userdata
        u = userdata.get("KAGGLE_USERNAME")
        k = userdata.get("KAGGLE_KEY")
        if u and k:
            os.environ["KAGGLE_USERNAME"] = u
            os.environ["KAGGLE_KEY"] = k
            return "colab-secrets"
    except Exception as e:
        print(f"  debug: Colab secrets unavailable ({type(e).__name__}) - expected "
              "under nbconvert. Provide /content/kaggle.json instead.")

    print("  no Kaggle credentials in /content/kaggle.json, environment, or Colab secrets")
    return None


def import_kaggle_api():
    import importlib
    import inspect
    # The kaggle package moved KaggleApi between versions:
    #   1.5.x  -> kaggle.api.api_factory.KaggleApi
    #   1.6+   -> kaggle.api.kaggle_api_extended.KaggleApi
    # Some of these modules also expose a same-named submodule, so verify the
    # resolved attribute is actually a class before using it.
    candidates = [
        "kaggle.api.kaggle_api_extended",
        "kaggle.api.api_factory",
        "kaggle",
    ]
    errors = []
    for module_name in candidates:
        try:
            mod = importlib.import_module(module_name)
        except Exception as e:
            errors.append(f"import {module_name}: {type(e).__name__}: {e}")
            continue
        for attr in ("KaggleApi", "KaggleApiExtended"):
            api_cls = getattr(mod, attr, None)
            if api_cls is None:
                continue
            if not inspect.isclass(api_cls):
                errors.append(f"{module_name}.{attr} is {type(api_cls).__name__}, not a class")
                continue
            print(f"  kaggle API from {module_name}.{attr}")
            return api_cls
        errors.append(f"{module_name}: no usable KaggleApi class attribute")
    raise ImportError("could not import KaggleApi from any known location:\n  "
                      + "\n  ".join(errors))


def download_from_kaggle():
    cred = kaggle_credentials_available()
    if not cred:
        print("  no Kaggle credentials available")
        return None
    print(f"  credentials source: {cred}")

    import importlib
    importlib.invalidate_caches()
    try:
        KaggleApi = import_kaggle_api()
    except ImportError:
        print("  installing kaggle package ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"],
                       check=True)
        importlib.invalidate_caches()
        KaggleApi = import_kaggle_api()

    api = KaggleApi()
    api.authenticate()
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    print(f"  downloading {KAGGLE_DATASET} -> {DATA_RAW}")
    api.dataset_download_files(KAGGLE_DATASET, path=str(DATA_RAW), unzip=True)
    root = find_class_root()
    if root is None:
        # the archive may unpack without the BSM/ level directory
        for cand in [DATA_RAW] + [p for p in DATA_RAW.glob("**/*") if p.is_dir()][:100]:
            if not cand.is_dir():
                continue
            kids = {d.name.strip() for d in cand.iterdir() if d.is_dir()}
            if len(kids & set(CLASSES)) >= 4:
                print(f"  resolved class root to {cand}")
                return cand
    return root


banner(2, "Dataset discovery")
CLASS_ROOT = find_class_root()
if CLASS_ROOT is None:
    print("  no local copy found, attempting Kaggle download ...")
    CLASS_ROOT = download_from_kaggle()

if CLASS_ROOT is None:
    raise SystemExit(
        "Dataset unavailable - training cannot be executed yet.\n"
        f"Place the six class folders under {DATA_RAW}, or provide Kaggle "
        "credentials (Colab secret KAGGLE_USERNAME / KAGGLE_KEY, or an uploaded "
        "/content/kaggle.json)."
    )

print("  class root :", CLASS_ROOT)
print("  folders    :", sorted(d.name for d in CLASS_ROOT.iterdir() if d.is_dir()))


  STEP 2. Dataset discovery
  no local copy found, attempting Kaggle download ...
  credentials source: /content/kaggle.json


  kaggle API from kaggle.api.kaggle_api_extended.KaggleApi
  downloading prasannavenkatesant/beach-sand-mineral-bsm -> /content/BSM-Project/data/raw
Dataset URL: https://www.kaggle.com/datasets/prasannavenkatesant/beach-sand-mineral-bsm


  class root : /content/BSM-Project/data/raw/BSM
  folders    : ['Garnet', 'Ilmenite', 'Monazite', 'Rutile', 'Sillimanite', 'Zircon']


---
## 3. Dataset inventory

Every file on disk is recorded — including files that later stages may reject —
so that exclusions are always traceable. Nothing is silently deleted.

In [3]:
banner(3, "Raw dataset inventory")

inventory_rows = []
for path in sorted(CLASS_ROOT.rglob("*")):
    if not path.is_file():
        continue
    parent = path.parent.name
    inventory_rows.append({
        "absolute_path": str(path),
        "relative_path": str(path.relative_to(CLASS_ROOT)),
        "folder_name_raw": parent,
        "file_name": path.name,
        "extension": path.suffix.lower(),
        "file_size_bytes": path.stat().st_size,
    })

raw_inventory = pd.DataFrame(inventory_rows)
if raw_inventory.empty:
    raise SystemExit("Dataset unavailable - no image files were found.")

raw_inventory["source_image_id"] = [
    f"IMG_{i:04d}" for i in range(len(raw_inventory))
]
raw_inventory = raw_inventory[[
    "source_image_id", "relative_path", "folder_name_raw", "file_name",
    "extension", "file_size_bytes", "absolute_path",
]]

save_csv(raw_inventory, "dataset_summary", "raw_dataset_inventory.csv")
raw_inventory.head(10)


  STEP 3. Raw dataset inventory
  wrote results/dataset_summary/raw_dataset_inventory.csv  (367 rows)


,source_image_id,relative_path,folder_name_raw,file_name,extension,file_size_bytes,absolute_path
0,IMG_0000,Garnet/1.jpg,Garnet,1.jpg,.jpg,29795,/content/BSM-Project/data/raw/BSM/Garnet/1.jpg
1,IMG_0001,Garnet/10.jpg,Garnet,10.jpg,.jpg,41585,/content/BSM-Project/data/raw/BSM/Garnet/10.jpg
2,IMG_0002,Garnet/11.jpg,Garnet,11.jpg,.jpg,14546,/content/BSM-Project/data/raw/BSM/Garnet/11.jpg
3,IMG_0003,Garnet/12.jpg,Garnet,12.jpg,.jpg,17174,/content/BSM-Project/data/raw/BSM/Garnet/12.jpg
4,IMG_0004,Garnet/13.jpg,Garnet,13.jpg,.jpg,23250,/content/BSM-Project/data/raw/BSM/Garnet/13.jpg
5,IMG_0005,Garnet/14.jpg,Garnet,14.jpg,.jpg,21121,/content/BSM-Project/data/raw/BSM/Garnet/14.jpg
6,IMG_0006,Garnet/15.jpg,Garnet,15.jpg,.jpg,16831,/content/BSM-Project/data/raw/BSM/Garnet/15.jpg
7,IMG_0007,Garnet/16.jpg,Garnet,16.jpg,.jpg,6971,/content/BSM-Project/data/raw/BSM/Garnet/16.jpg
8,IMG_0008,Garnet/17.jpg,Garnet,17.jpg,.jpg,25417,/content/BSM-Project/data/raw/BSM/Garnet/17.jpg
9,IMG_0009,Garnet/18.jpg,Garnet,18.jpg,.jpg,17311,/content/BSM-Project/data/raw/BSM/Garnet/18.jpg


---
## 4–5. Image validation and corrupt-image detection

Each file is opened and fully decoded. `Image.verify()` catches truncated
headers; a real decode + `convert("RGB")` catches pixel-level corruption.
Both checks are recorded, not assumed.

In [4]:
banner(4, "Image validation / corrupt detection")

val_rows = []
for _, row in raw_inventory.iterrows():
    p = Path(row["absolute_path"])
    rec = dict(row)
    rec.update({
        "is_readable": False,
        "verify_ok": False,
        "decode_ok": False,
        "width": np.nan,
        "height": np.nan,
        "mode_raw": None,
        "channels": np.nan,
        "megapixels": np.nan,
        "aspect_ratio": np.nan,
        "error": "",
    })
    try:
        with Image.open(p) as im:
            rec["mode_raw"] = im.mode
            rec["width"] = im.size[0]
            rec["height"] = im.size[1]
            rec["verify_ok"] = True
        with Image.open(p) as im:
            im.load()
            rec["decode_ok"] = True
            rec["is_readable"] = True
            rgb = im.convert("RGB")
            rec["channels"] = len(rgb.getbands())
            arr = np.asarray(rgb)
        rec["megapixels"] = round(rec["width"] * rec["height"] / 1e6, 4)
        rec["aspect_ratio"] = round(rec["width"] / rec["height"], 4)
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:200]
    val_rows.append(rec)

image_report = pd.DataFrame(val_rows)

n_total = len(image_report)
n_readable = int(image_report["is_readable"].sum())
n_verify_fail = int((~image_report["verify_ok"]).sum())
n_decode_fail = int((~image_report["decode_ok"]).sum())
n_unsupported = int((~image_report["extension"].isin(SUPPORTED_EXT)).sum())

print(f"  files discovered      : {n_total}")
print(f"  fully readable        : {n_readable}")
print(f"  header-verify failures: {n_verify_fail}")
print(f"  decode failures       : {n_decode_fail}")
print(f"  unsupported extension : {n_unsupported}")

if n_decode_fail or n_unsupported:
    print("\n  files failing validation:")
    bad = image_report[~image_report["is_readable"] | ~image_report["extension"].isin(SUPPORTED_EXT)]
    print(bad[["source_image_id", "relative_path", "error"]].to_string(index=False))


  STEP 4. Image validation / corrupt detection


  files discovered      : 367
  fully readable        : 367
  header-verify failures: 0
  decode failures       : 0
  unsupported extension : 0


---
## 6. Dimension / channel analysis

In [5]:
banner(6, "Dimension and channel analysis")

print("  modes   :", image_report["mode_raw"].value_counts().to_dict())
print("  channels:", image_report["channels"].value_counts().to_dict())
print()
dim_desc = image_report[["width", "height", "channels", "megapixels", "aspect_ratio"]].describe().T
print(dim_desc.round(3).to_string())

smaller_side = image_report[["width", "height"]].min(axis=1)
print(f"\n  min dimension across dataset : {int(smaller_side.min())} px")
print(f"  max dimension across dataset : {int(image_report[['width','height']].max().max())} px")
print(f"  images below {IMG_SIZE}px on the short side : {int((smaller_side < IMG_SIZE).sum())}")
print("\n  -> direct resize to %dx%d; no centre crop (crops would discard mineral grains)." % (IMG_SIZE, IMG_SIZE))


  STEP 6. Dimension and channel analysis
  modes   : {'RGB': 367}
  channels: {3: 367}

              count     mean     std      min      25%      50%      75%     max
width         367.0  277.659  95.669  102.000  215.500  253.000  323.000  765.00
height        367.0  218.469  69.714   91.000  173.500  192.000  251.000  641.00
channels      367.0    3.000   0.000    3.000    3.000    3.000    3.000    3.00
megapixels    367.0    0.066   0.050    0.014    0.038    0.049    0.078    0.48
aspect_ratio  367.0    1.291   0.288    0.423    1.113    1.302    1.471    2.34

  min dimension across dataset : 91 px
  max dimension across dataset : 765 px
  images below 224px on the short side : 246

  -> direct resize to 224x224; no centre crop (crops would discard mineral grains).


---
## 7. Duplicate detection

Exact duplicates are found by SHA-256 of the raw bytes. Exact-duplicate images
form a **duplicate group**: every member of a group shares one `group_id`, and
a group is guaranteed to land entirely inside a single split later on. That is
the structural guarantee against leakage.

In [6]:
banner(7, "Exact duplicate detection")

hashes = {}
for _, row in image_report.iterrows():
    hashes[row["source_image_id"]] = sha256_of(row["absolute_path"])
image_report["sha256"] = image_report["source_image_id"].map(hashes)

n_unique_hashes = image_report["sha256"].nunique()
n_dup_rows = int(image_report.duplicated("sha256").sum())
print(f"  images                  : {len(image_report)}")
print(f"  unique sha256 hashes    : {n_unique_hashes}")
print(f"  redundant copies        : {n_dup_rows}")

if n_dup_rows:
    dup_hashes = image_report["sha256"][image_report.duplicated("sha256", keep=False)]
    print("\n  duplicate clusters:")
    print(image_report[image_report["sha256"].isin(set(dup_hashes))][
        ["source_image_id", "relative_path", "sha256"]].to_string(index=False))
else:
    print("  no exact duplicates found")

# group_id: exact-duplicate images share a group; unique images are their own group.
first_seen = {}
group_ids = []
for _, row in image_report.iterrows():
    key = row["sha256"]
    if key not in first_seen:
        first_seen[key] = f"GRP_{len(first_seen):04d}"
    group_ids.append(first_seen[key])
image_report["group_id"] = group_ids

sizes = image_report.groupby("group_id").size()
print(f"\n  groups                  : {sizes.size}")
print(f"  largest group            : {int(sizes.max())} image(s)")
print(f"  multi-image groups       : {int((sizes > 1).sum())}")


  STEP 7. Exact duplicate detection
  images                  : 367
  unique sha256 hashes    : 367
  redundant copies        : 0
  no exact duplicates found

  groups                  : 367
  largest group            : 1 image(s)
  multi-image groups       : 0


---
## 8. Class-name normalisation

In [7]:
banner(8, "Class-name normalisation")

CLASS_ALIASES = {
    "ilmenite": "Ilmenite", "ilmenites": "Ilmenite", "ilmenite_": "Ilmenite",
    "rutile": "Rutile", "rutiles": "Rutile",
    "zircon": "Zircon", "zircons": "Zircon",
    "garnet": "Garnet", "garnets": "Garnet",
    "monazite": "Monazite", "monazites": "Monazite",
    "sillimanite": "Sillimanite", "sillimanites": "Sillimanite",
    "sillimanite_": "Sillimanite", "sillimanite ": "Sillimanite",
}

def normalize_class(raw_name):
    key = str(raw_name).strip().lower()
    key = key.replace(" ", "_").replace("-", "_").replace(".", "")
    if key in CLASS_ALIASES:
        return CLASS_ALIASES[key]
    key2 = key.rstrip("_")
    return CLASS_ALIASES.get(key2, None)

image_report["class_raw"] = image_report["folder_name_raw"]
image_report["class"] = image_report["class_raw"].map(normalize_class)
image_report["class_mapped"] = image_report["class"].notna()

unknown = image_report.loc[~image_report["class_mapped"], "class_raw"].unique().tolist()
print("  folders found      :", sorted(image_report["class_raw"].unique().tolist()))
print("  unmapped folders   :", unknown if unknown else "none")

missing = sorted(set(CLASSES) - set(image_report.loc[image_report["class_mapped"], "class"]))
print("  expected classes   :", CLASSES)
print("  classes not found  :", missing if missing else "none")
if missing:
    raise SystemExit(f"Dataset incomplete - expected classes missing from disk: {missing}")


  STEP 8. Class-name normalisation
  folders found      : ['Garnet', 'Ilmenite', 'Monazite', 'Rutile', 'Sillimanite', 'Zircon']
  unmapped folders   : none
  expected classes   : ['Ilmenite', 'Rutile', 'Zircon', 'Garnet', 'Monazite', 'Sillimanite']
  classes not found  : none


---
## 9. Data wrangling

Combines validation, extension filtering, class normalisation, duplicate-path
detection, exact-duplicate detection and corrupt-image detection into one
auditable decision per image. Every exclusion is reported with a reason; nothing
is deleted without a record.

In [8]:
banner(9, "Data wrangling")

wrangling_rows = []
keep = []
seen_paths = set()

for _, row in image_report.iterrows():
    reasons = []
    apath = row["absolute_path"]

    if apath in seen_paths:
        reasons.append("duplicate_path")
    seen_paths.add(apath)

    if not row["class_mapped"]:
        reasons.append("unmapped_class_name")
    if row["extension"] not in SUPPORTED_EXT:
        reasons.append("unsupported_extension")
    if not row["verify_ok"]:
        reasons.append("header_verify_failed")
    if not row["decode_ok"]:
        reasons.append("corrupt_or_undecodable")
    if pd.isna(row["width"]) or pd.isna(row["height"]) or row["width"] <= 0 or row["height"] <= 0:
        reasons.append("invalid_dimensions")

    kept = len(reasons) == 0
    wrangling_rows.append({
        "source_image_id": row["source_image_id"],
        "relative_path": row["relative_path"],
        "absolute_path": row["absolute_path"],
        "class_raw": row["class_raw"],
        "class": row["class"] if row["class_mapped"] else None,
        "group_id": row["group_id"],
        "extension": row["extension"],
        "file_size_bytes": row["file_size_bytes"],
        "width": row["width"],
        "height": row["height"],
        "channels": row["channels"],
        "megapixels": row["megapixels"],
        "aspect_ratio": row["aspect_ratio"],
        "sha256": row["sha256"],
        "is_valid": kept,
        "exclusion_reason": None if kept else ";".join(reasons),
    })
    if kept:
        keep.append(row["source_image_id"])

wrangling = pd.DataFrame(wrangling_rows)
clean = wrangling[wrangling["is_valid"]].copy().reset_index(drop=True)

save_csv(wrangling, "preprocessing", "data_wrangling_report.csv")

print(f"\n  input files          : {len(wrangling)}")
print(f"  retained             : {len(clean)}")
print(f"  excluded             : {int((~wrangling['is_valid']).sum())}")
if (~wrangling["is_valid"]).sum():
    print("\n  exclusion reasons:")
    print(wrangling.loc[~wrangling["is_valid"], "exclusion_reason"].value_counts().to_string())

clean = clean.sort_values("source_image_id").reset_index(drop=True)
save_csv(clean, "preprocessing", "clean_dataset_manifest.csv")


  STEP 9. Data wrangling
  wrote results/preprocessing/data_wrangling_report.csv  (367 rows)

  input files          : 367
  retained             : 367
  excluded             : 0
  wrote results/preprocessing/clean_dataset_manifest.csv  (367 rows)


PosixPath('/content/BSM-Project/results/preprocessing/clean_dataset_manifest.csv')

---
## 10. Metadata imputation (median)

The paper describes median-based imputation for missing / null / outlier
metadata. For an image dataset the **practical interpretation** is to impute
*image metadata fields* — width, height, channels and file size — never pixel
values. Inventing pixel-level imputation for a classification task is not
supported by the paper and is not done here.

In [9]:
banner(10, "Metadata imputation (median)")

IMPUTE_FIELDS = ["width", "height", "channels", "file_size_bytes"]
imputation_rows = []

for field in IMPUTE_FIELDS:
    values = pd.to_numeric(clean[field], errors="coerce")
    median_value = float(values.median()) if values.notna().any() else np.nan

    missing_mask = values.isna()
    # An outlier for these fields is a physically impossible value.
    outlier_mask = pd.Series(False, index=clean.index)
    if field in ("width", "height"):
        outlier_mask = values.notna() & (values <= 0)
    elif field == "channels":
        outlier_mask = values.notna() & ~values.isin([1, 3, 4])
    elif field == "file_size_bytes":
        outlier_mask = values.notna() & (values <= 0)

    needs_impute = missing_mask | outlier_mask
    n_imputed = int(needs_impute.sum())

    imputation_rows.append({
        "field": field,
        "n_records": int(len(clean)),
        "n_missing": int(missing_mask.sum()),
        "n_outliers": int(outlier_mask.sum()),
        "n_imputed": n_imputed,
        "median_value": round(median_value, 4) if pd.notna(median_value) else None,
        "min_before": float(values.min()) if values.notna().any() else None,
        "max_before": float(values.max()) if values.notna().any() else None,
        "strategy": "median",
        "note": ("no missing or invalid values required imputation"
                 if n_imputed == 0 else "median substituted"),
    })

    if n_imputed:
        clean.loc[needs_impute, field] = median_value
        for _, r in clean[needs_impute].iterrows():
            imputation_rows.append({
                "field": field,
                "n_records": int(len(clean)),
                "n_missing": int(missing_mask.sum()),
                "n_outliers": int(outlier_mask.sum()),
                "n_imputed": n_imputed,
                "median_value": round(median_value, 4),
                "min_before": None,
                "max_before": None,
                "strategy": "median",
                "note": f"imputed {field} for {r['source_image_id']} ({r['relative_path']})",
            })

imputation_report = pd.DataFrame(imputation_rows)
save_csv(imputation_report, "preprocessing", "data_imputation_report.csv")
print(imputation_report[imputation_report["field"].isin(IMPUTE_FIELDS)].to_string(index=False))


  STEP 10. Metadata imputation (median)
  wrote results/preprocessing/data_imputation_report.csv  (4 rows)
          field  n_records  n_missing  n_outliers  n_imputed  median_value  min_before  max_before strategy                                             note
          width        367          0           0          0         253.0       102.0       765.0   median no missing or invalid values required imputation
         height        367          0           0          0         192.0        91.0       641.0   median no missing or invalid values required imputation
       channels        367          0           0          0           3.0         3.0         3.0   median no missing or invalid values required imputation
file_size_bytes        367          0           0          0       14588.0      3020.0    168769.0   median no missing or invalid values required imputation


---
## 11. EDA (concise)

In [10]:
banner(11, "Exploratory data analysis")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

actual = clean["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)
actual_df = pd.DataFrame({
    "class": CLASSES,
    "actual_count": actual.values,
    "paper_reported_count": [PAPER_CLASS_COUNTS[c] for c in CLASSES],
})
actual_df["difference"] = actual_df["actual_count"] - actual_df["paper_reported_count"]
actual_df["paper_total_row_sum"] = PAPER_REPORTED_IMAGES

dataset_summary = actual_df.copy()
dataset_summary.loc[len(dataset_summary)] = {
    "class": "TOTAL",
    "actual_count": int(actual.sum()),
    "paper_reported_count": sum(PAPER_CLASS_COUNTS.values()),
    "difference": int(actual.sum()) - sum(PAPER_CLASS_COUNTS.values()),
    "paper_total_row_sum": PAPER_REPORTED_IMAGES,
}
save_csv(dataset_summary, "dataset_summary", "dataset_summary.csv")
print(dataset_summary.to_string(index=False))
print(f"\n  paper text states {PAPER_REPORTED_IMAGES} images, but the paper's own six")
print(f"  class counts sum to {sum(PAPER_CLASS_COUNTS.values())}. This implementation")
print(f"  reports the ACTUAL {int(actual.sum())} images found on disk and never")
print("  forces the total to match the paper.")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
x = np.arange(len(CLASSES))
ax[0].bar(x - 0.2, actual.values, 0.4, label="Our dataset (actual)", color="#4C72B0")
ax[0].bar(x + 0.2, [PAPER_CLASS_COUNTS[c] for c in CLASSES], 0.4,
          label="Paper reported", color="#DD8452", alpha=0.85)
ax[0].set_xticks(x); ax[0].set_xticklabels(CLASSES, rotation=20, ha="right")
ax[0].set_ylabel("images"); ax[0].set_title("Class distribution: ours vs paper")
ax[0].legend(); ax[0].grid(alpha=0.3, axis="y")

ax[1].bar(x, actual.values, color="#55A868")
for i, v in enumerate(actual.values):
    ax[1].text(i, v + 0.6, str(int(v)), ha="center", fontsize=9)
ax[1].set_xticks(x); ax[1].set_xticklabels(CLASSES, rotation=20, ha="right")
ax[1].set_ylabel("images"); ax[1].set_title("Actual class distribution")
ax[1].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(R("dataset_summary") / "class_distribution.png", dpi=150); plt.close()
print("  wrote results/dataset_summary/class_distribution.png")


  STEP 11. Exploratory data analysis


  wrote results/dataset_summary/dataset_summary.csv  (7 rows)
      class  actual_count  paper_reported_count  difference  paper_total_row_sum
   Ilmenite            62                    60           2                  357
     Rutile            66                    62           4                  357
     Zircon            59                    60          -1                  357
     Garnet            60                    66          -6                  357
   Monazite            60                    60           0                  357
Sillimanite            60                    59           1                  357
      TOTAL           367                   367           0                  357

  paper text states 357 images, but the paper's own six
  class counts sum to 367. This implementation
  reports the ACTUAL 367 images found on disk and never
  forces the total to match the paper.


  wrote results/dataset_summary/class_distribution.png


In [11]:
# Sample images per class
rng = np.random.default_rng(SEED)
n_show = 5
fig, axes = plt.subplots(len(CLASSES), n_show, figsize=(14, 2.3 * len(CLASSES)))
for ci, cls in enumerate(CLASSES):
    sub = clean[clean["class"] == cls]
    picks = rng.choice(len(sub), size=min(n_show, len(sub)), replace=False)
    for pi, idx in enumerate(picks):
        ax = axes[ci, pi]
        row = clean.loc[[idx]].iloc[0]
        try:
            im = Image.open(row["absolute_path"]).convert("RGB")
            ax.imshow(im)
            ax.set_title(f"{int(row['width'])}x{int(row['height'])}", fontsize=7)
        except Exception:
            ax.text(0.5, 0.5, "unreadable", ha="center", va="center")
        ax.axis("off")
        if pi == 0:
            ax.text(-0.06, 0.5, cls, transform=ax.transAxes, rotation=90,
                    va="center", ha="right", fontsize=11, fontweight="bold")
plt.suptitle(f"Sample images — {n_show} per class (n = {len(clean)})", fontsize=13)
plt.tight_layout(); plt.savefig(R("dataset_summary") / "sample_images.png", dpi=130); plt.close()
print("  wrote results/dataset_summary/sample_images.png")

  wrote results/dataset_summary/sample_images.png


In [12]:
# Image dimensions
palette = plt.get_cmap("tab10")
cmap = {c: palette(i) for i, c in enumerate(CLASSES)}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for cls in CLASSES:
    sub = clean[clean["class"] == cls]
    axes[0].scatter(sub["width"], sub["height"], s=26, alpha=0.75,
                    label=cls, color=cmap[cls], edgecolors="none")
axes[0].axvline(IMG_SIZE, ls="--", c="k", lw=1, alpha=0.6)
axes[0].axhline(IMG_SIZE, ls="--", c="k", lw=1, alpha=0.6)
axes[0].set_xlabel("width (px)"); axes[0].set_ylabel("height (px)")
axes[0].set_title(f"Image dimensions (dashed = {IMG_SIZE})")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

for cls in CLASSES:
    sub = clean[clean["class"] == cls]
    axes[1].hist(sub["megapixels"], bins=12, alpha=0.55, label=cls, color=cmap[cls])
axes[1].set_xlabel("megapixels"); axes[1].set_ylabel("images")
axes[1].set_title("Resolution distribution"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.savefig(R("dataset_summary") / "image_dimensions.png", dpi=150); plt.close()
print("  wrote results/dataset_summary/image_dimensions.png")

  wrote results/dataset_summary/image_dimensions.png


---
## 12–14. Source-group split (before augmentation)

**The leakage safeguard.** Every original image belongs to a `group_id`
(exact-duplicate images share a group). The 70/15/15 split is performed over
*groups*, stratified by class, with seed 42. It is therefore structurally
impossible for a duplicate of a training image to appear in validation or test.

Augmentation is applied **after** the split and **only to the training split**,
so validation and test are evaluated on clean, un-augmented originals.

In [13]:
banner(12, "Source-group level split")

SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
assert abs(sum(SPLIT_RATIOS.values()) - 1.0) < 1e-9


def stratified_group_split(df, ratios=None, seed=SEED):
    ratios = ratios or SPLIT_RATIOS
    rng = np.random.default_rng(seed)
    groups = df.groupby("group_id")["class"].first().reset_index()
    assignment = {}
    for cls in sorted(groups["class"].unique()):
        members = groups.loc[groups["class"] == cls, "group_id"].tolist()
        order = list(rng.permutation(np.array(members, dtype=object)))
        n = len(order)
        n_train = int(round(n * ratios["train"]))
        n_val = int(round(n * ratios["val"]))
        # guarantee a non-empty val and test when the class has enough groups
        if n - n_train - n_val <= 0:
            n_val = max(1, n - n_train - 1)
        for i, gid in enumerate(order):
            if i < n_train:
                assignment[gid] = "train"
            elif i < n_train + n_val:
                assignment[gid] = "val"
            else:
                assignment[gid] = "test"
    return df["group_id"].map(assignment)


clean["split"] = stratified_group_split(clean)

split_counts = (clean.groupby(["split", "class"]).size().unstack(fill_value=0)
                .reindex(columns=CLASSES).reindex(["train", "val", "test"]).fillna(0).astype(int))
per_split_total = split_counts.sum(axis=1)
split_counts["TOTAL"] = per_split_total
split_counts.loc["TOTAL"] = per_split_total.sum()
print(split_counts.to_string())

# ---- leakage assertions -------------------------------------------------
groups_per_split = clean.groupby("group_id")["split"].nunique()
n_shared = int((groups_per_split > 1).sum())
print(f"\n  groups spanning more than one split : {n_shared}")

train_groups = set(clean.loc[clean["split"] == "train", "group_id"])
val_groups = set(clean.loc[clean["split"] == "val", "group_id"])
test_groups = set(clean.loc[clean["split"] == "test", "group_id"])

assert n_shared == 0, "LEAKAGE: a group appears in more than one split"
assert not (train_groups & val_groups), "LEAKAGE: group shared between train and val"
assert not (train_groups & test_groups), "LEAKAGE: group shared between train and test"
assert not (val_groups & test_groups), "LEAKAGE: group shared between val and test"
print("  LEAKAGE CHECK PASSED: no group appears in more than one split.")


  STEP 12. Source-group level split
class  Ilmenite  Rutile  Zircon  Garnet  Monazite  Sillimanite  TOTAL
split                                                                
train        43      46      41      42        42           42    256
val           9      10       9       9         9            9     55
test         10      10       9       9         9            9     56
TOTAL       367     367     367     367       367          367    367

  groups spanning more than one split : 0
  LEAKAGE CHECK PASSED: no group appears in more than one split.


---
## 15. Augmentation (training split only)

Paper-described transforms: **horizontal inversion (flip)** and **rotation**
at `0°, 30°, 45°, 90°, 180°, 270°`.

The paper reports a final augmented dataset of **≈1468 images**. That number is
recorded as a reference target only. This implementation generates exactly the
transforms the paper describes and reports what it actually produced, with the
difference explained. It does **not** force the count to 1468.

In [14]:
banner(13, "Augmentation - TRAIN SPLIT ONLY")

AUG_ROTATIONS = [0, 30, 45, 90, 180, 270]
AUG_HFLIP = [False, True]

DATA_PROC.mkdir(parents=True, exist_ok=True)
DATA_AUG.mkdir(parents=True, exist_ok=True)

abs_by_id = dict(zip(raw_inventory["source_image_id"], raw_inventory["absolute_path"]))

# --- 14a. processed 224x224 copies for val / test -----------------------
proc_rows = []
for _, row in clean.iterrows():
    dst_dir = DATA_PROC / row["class"]
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / (Path(row["relative_path"]).stem + ".jpg")
    with Image.open(abs_by_id[row["source_image_id"]]) as im:
        im = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
        im.save(dst, "JPEG", quality=95)
    proc_rows.append({**row.to_dict(), "processed_path": str(dst)})

processed = pd.DataFrame(proc_rows)
print(f"  wrote {len(processed)} processed 224x224 images -> {DATA_PROC.relative_to(PROJECT_ROOT)}")


  STEP 13. Augmentation - TRAIN SPLIT ONLY


  wrote 367 processed 224x224 images -> data/processed


In [15]:
# --- 14b. augmented variants, training split only ------------------------
aug_rows = []
train_clean = clean[clean["split"] == "train"]
print(f"  augmenting {len(train_clean)} training source images "
      f"x {len(AUG_ROTATIONS)} rotations x {len(AUG_HFLIP)} flip states")

for _, row in train_clean.iterrows():
    src = abs_by_id[row["source_image_id"]]
    dst_dir = DATA_AUG / "train" / row["class"]
    dst_dir.mkdir(parents=True, exist_ok=True)
    with Image.open(src) as im:
        original = im.convert("RGB")
    stem = Path(row["relative_path"]).stem
    for rot in AUG_ROTATIONS:
        rot_im = original.rotate(rot, expand=False, resample=Image.BICUBIC) if rot else original.copy()
        for flip in AUG_HFLIP:
            out = rot_im.transpose(Image.FLIP_LEFT_RIGHT) if flip else rot_im
            out = out.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
            aug_id = f"{row['source_image_id']}_r{rot}_f{int(flip)}"
            dst = dst_dir / f"{stem}__r{rot}_f{int(flip)}.jpg"
            out.save(dst, "JPEG", quality=95)
            aug_rows.append({
                "source_image_id": row["source_image_id"],
                "source_path": row["absolute_path"],
                "class": row["class"],
                "group_id": row["group_id"],
                "split": "train",
                "augmentation_id": aug_id,
                "rotation_angle": rot,
                "horizontal_flip": bool(flip),
                "augmented_path": str(dst),
            })

aug_manifest = pd.DataFrame(aug_rows)
save_csv(aug_manifest, "augmentation", "augmentation_manifest.csv")
print(f"  produced {len(aug_manifest)} augmented training images "
      f"-> {DATA_AUG.relative_to(PROJECT_ROOT)}")

  augmenting 256 training source images x 6 rotations x 2 flip states


  wrote results/augmentation/augmentation_manifest.csv  (3072 rows)
  produced 3072 augmented training images -> data/augmented


In [16]:
# --- 14c. augmentation summary + paper comparison ------------------------
per_class_aug = aug_manifest["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)

aug_summary = pd.DataFrame({
    "class": CLASSES,
    "source_images": [int((train_clean["class"] == c).sum()) for c in CLASSES],
    "rotations_per_image": len(AUG_ROTATIONS),
    "flip_states_per_rotation": len(AUG_HFLIP),
    "variants_per_source": len(AUG_ROTATIONS) * len(AUG_HFLIP),
    "augmented_images": per_class_aug.values,
})
aug_summary.loc[len(aug_summary)] = {
    "class": "TOTAL",
    "source_images": int(aug_summary["source_images"].sum()),
    "rotations_per_image": len(AUG_ROTATIONS),
    "flip_states_per_rotation": len(AUG_HFLIP),
    "variants_per_source": len(AUG_ROTATIONS) * len(AUG_HFLIP),
    "augmented_images": int(aug_summary["augmented_images"].sum()),
}
save_csv(aug_summary, "augmentation", "augmentation_summary.csv")
print(aug_summary.to_string(index=False))

actual_aug = len(aug_manifest)
notes = pd.DataFrame([
    {"quantity": "paper_reported_total_images", "value": PAPER_REPORTED_IMAGES,
     "note": "Paper text states 357; the paper's own six class counts sum to 367."},
    {"quantity": "paper_reported_augmented_images", "value": PAPER_REPORTED_AUGMENTED,
     "note": "Paper reports ~1468 = 367 x 4. Reference target only, never forced."},
    {"quantity": "our_actual_total_images", "value": int(len(clean)),
     "note": "Images actually present and valid after wrangling."},
    {"quantity": "our_source_images_used_for_training", "value": int(len(train_clean)),
     "note": "70% group-level training split."},
    {"quantity": "our_actual_augmented_images", "value": actual_aug,
     "note": f"{len(train_clean)} sources x {len(AUG_ROTATIONS)} rotations "
             f"x {len(AUG_HFLIP)} flip states."},
    {"quantity": "difference_vs_paper_augmented", "value": actual_aug - PAPER_REPORTED_AUGMENTED,
     "note": "Differs because (a) augmentation is applied to the 70% training split "
             "only, not the whole dataset, and (b) each rotation is paired with both "
             "flip states, giving 12 variants per source rather than 4. The paper "
             "does not state how augmented variants were split, so its 1468 cannot "
             "be matched without reintroducing train/test leakage."},
])
save_csv(notes, "augmentation", "augmentation_count_comparison.csv")
print("\n" + notes.to_string(index=False))

  wrote results/augmentation/augmentation_summary.csv  (7 rows)
      class  source_images  rotations_per_image  flip_states_per_rotation  variants_per_source  augmented_images
   Ilmenite             43                    6                         2                   12               516
     Rutile             46                    6                         2                   12               552
     Zircon             41                    6                         2                   12               492
     Garnet             42                    6                         2                   12               504
   Monazite             42                    6                         2                   12               504
Sillimanite             42                    6                         2                   12               504
      TOTAL            256                    6                         2                   12              3072
  wrote results/augmentation/aug

In [17]:
# --- 14d. augmentation examples ------------------------------------------
fig, axes = plt.subplots(3, len(AUG_ROTATIONS), figsize=(3.0 * len(AUG_ROTATIONS), 6.4))
demo = train_clean.iloc[0] if len(train_clean) else clean.iloc[0]
src_path = abs_by_id[demo["source_image_id"]]
with Image.open(src_path) as im:
    demo_img = im.convert("RGB")
for ci, rot in enumerate(AUG_ROTATIONS):
    r = demo_img.rotate(rot, expand=False, resample=Image.BICUBIC) if rot else demo_img.copy()
    for ri, flip in enumerate(AUG_HFLIP):
        ax = axes[ri, ci]
        out = r.transpose(Image.FLIP_LEFT_RIGHT) if flip else r
        ax.imshow(out.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))
        ax.set_title(f"rot {rot}°\nflip={flip}", fontsize=9)
        ax.axis("off")
fig.suptitle(f"Augmentation examples — source {demo['source_image_id']} ({demo['class']})", fontsize=12)
plt.tight_layout(); plt.savefig(R("augmentation") / "augmentation_examples.png", dpi=130); plt.close()
print("  wrote results/augmentation/augmentation_examples.png")

  wrote results/augmentation/augmentation_examples.png


---
## 16. Final dataset manifest

One row per **sample actually used** by training or evaluation: augmented
training samples plus clean processed validation/test samples.

In [18]:
banner(14, "Final dataset manifest")

train_rows = []
for _, a in aug_manifest.iterrows():
    p = Path(a["augmented_path"])
    train_rows.append({
        "sample_id": a["augmentation_id"],
        "split": "train",
        "class": a["class"],
        "class_idx": CLASS_TO_IDX[a["class"]],
        "source_image_id": a["source_image_id"],
        "source_path": a["source_path"],
        "group_id": a["group_id"],
        "sample_path": a["augmented_path"],
        "is_augmented": True,
        "rotation_angle": a["rotation_angle"],
        "horizontal_flip": a["horizontal_flip"],
        "width": IMG_SIZE,
        "height": IMG_SIZE,
        "channels": 3,
        "file_size_bytes": p.stat().st_size,
        "sha256": sha256_of(p),
    })

eval_rows = []
for _, r in processed[processed["split"].isin(["val", "test"])].iterrows():
    p = Path(r["processed_path"])
    eval_rows.append({
        "sample_id": r["source_image_id"],
        "split": r["split"],
        "class": r["class"],
        "class_idx": CLASS_TO_IDX[r["class"]],
        "source_image_id": r["source_image_id"],
        "source_path": r["absolute_path"],
        "group_id": r["group_id"],
        "sample_path": r["processed_path"],
        "is_augmented": False,
        "rotation_angle": 0,
        "horizontal_flip": False,
        "width": IMG_SIZE,
        "height": IMG_SIZE,
        "channels": 3,
        "file_size_bytes": p.stat().st_size,
        "sha256": sha256_of(p),
    })

final_manifest = pd.concat([pd.DataFrame(train_rows), pd.DataFrame(eval_rows)], ignore_index=True)
final_manifest = final_manifest.sort_values(["split", "class", "sample_id"]).reset_index(drop=True)
save_csv(final_manifest, "preprocessing", "final_split_manifest.csv")

manifest_counts = final_manifest.groupby("split").size().reindex(["train", "val", "test"]).fillna(0).astype(int)
print("\n  samples per split:")
print(manifest_counts.to_string())
print("\n  distinct source images per split:")
print(final_manifest.groupby("split")["source_image_id"].nunique().reindex(["train", "val", "test"]).to_string())


  STEP 14. Final dataset manifest


  wrote results/preprocessing/final_split_manifest.csv  (3183 rows)

  samples per split:
split
train    3072
val        55
test       56

  distinct source images per split:
split
train    256
val       55
test      56


In [19]:
# ---- hard leakage assertions on the FINAL sample set --------------------
banner(15, "Final leakage verification")

src_by_split = final_manifest.groupby("split")["source_image_id"].apply(set)
overlap_tr_va = src_by_split["train"] & src_by_split["val"]
overlap_tr_te = src_by_split["train"] & src_by_split["test"]
overlap_va_te = src_by_split["val"] & src_by_split["test"]

val_aug = int(((final_manifest["split"] == "val") & final_manifest["is_augmented"]).sum())
test_aug = int(((final_manifest["split"] == "test") & final_manifest["is_augmented"]).sum())

print(f"  train/val shared source images : {len(overlap_tr_va)}")
print(f"  train/test shared source images: {len(overlap_tr_te)}")
print(f"  val/test shared source images  : {len(overlap_va_te)}")
print(f"  augmented samples in val       : {val_aug}")
print(f"  augmented samples in test      : {test_aug}")

assert len(overlap_tr_va) == 0 and len(overlap_tr_te) == 0 and len(overlap_va_te) == 0, \
    "LEAKAGE: source images shared across splits"
assert val_aug == 0 and test_aug == 0, "LEAKAGE: augmented samples found in val/test"
print("\n  DATA LEAKAGE CHECK PASSED")
print("  No original image contributes samples to more than one split,")
print("  and validation/test contain no augmented images.")


  STEP 15. Final leakage verification
  train/val shared source images : 0
  train/test shared source images: 0
  val/test shared source images  : 0
  augmented samples in val       : 0
  augmented samples in test      : 0

  DATA LEAKAGE CHECK PASSED
  No original image contributes samples to more than one split,
  and validation/test contain no augmented images.


---
## 17. Dataset version summary

A single fingerprint table: counts, config, library versions and SHA-256 of each
generated manifest. Any later change to the dataset changes this table.

In [20]:
banner(16, "Dataset version summary")

def git_commit():
    try:
        r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(PROJECT_ROOT),
                           capture_output=True, text=True, timeout=20)
        if r.returncode != 0:
            return "no-commits-yet"
        return r.stdout.strip() or "unknown"
    except Exception:
        return "unavailable"

import sklearn
version_rows = [
    ("timestamp_utc", datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")),
    ("git_commit", git_commit()),
    ("seed", SEED),
    ("project_root", str(PROJECT_ROOT)),
    ("class_root", str(CLASS_ROOT)),
    ("class_order", ", ".join(CLASSES)),
    ("n_classes", len(CLASSES)),
    ("image_size", f"{IMG_SIZE}x{IMG_SIZE}x3"),
    ("raw_files_discovered", len(raw_inventory)),
    ("files_retained_after_wrangling", len(clean)),
    ("files_excluded", int((~wrangling["is_valid"]).sum())),
    ("corrupt_or_undecodable", int((~image_report["decode_ok"]).sum())),
    ("exact_duplicate_redundant_copies", n_dup_rows),
    ("duplicate_groups", int(image_report.groupby("group_id").ngroups)),
    ("paper_reported_images", PAPER_REPORTED_IMAGES),
    ("paper_reported_augmented_images", PAPER_REPORTED_AUGMENTED),
    ("actual_augmented_images", int(len(aug_manifest))),
    ("augmentation_rotations", ", ".join(str(r) for r in AUG_ROTATIONS)),
    ("augmentation_flip_states", ", ".join(str(bool(f)) for f in AUG_HFLIP)),
    ("augmentation_scope", "train split only (leakage safeguard)"),
    ("split_ratios", "train 0.70 / val 0.15 / test 0.15"),
    ("split_unit", "source-image group_id"),
    ("split_seed", SEED),
    ("split_train_sources", int((clean["split"] == "train").sum())),
    ("split_val_sources", int((clean["split"] == "val").sum())),
    ("split_test_sources", int((clean["split"] == "test").sum())),
    ("samples_train", int((final_manifest["split"] == "train").sum())),
    ("samples_val", int((final_manifest["split"] == "val").sum())),
    ("samples_test", int((final_manifest["split"] == "test").sum())),
    ("leakage_check", "PASSED - no group spans splits; no augmented val/test samples"),
    ("sha256_raw_inventory", sha256_of(R("dataset_summary") / "raw_dataset_inventory.csv")),
    ("sha256_clean_manifest", sha256_of(R("preprocessing") / "clean_dataset_manifest.csv")),
    ("sha256_augmentation_manifest", sha256_of(R("augmentation") / "augmentation_manifest.csv")),
    ("sha256_final_split_manifest", sha256_of(R("preprocessing") / "final_split_manifest.csv")),
    ("python_version", platform.python_version()),
    ("numpy_version", np.__version__),
    ("pandas_version", pd.__version__),
    ("pillow_version", Image.__version__ if hasattr(Image, "__version__") else "n/a"),
    ("sklearn_version", sklearn.__version__),
]

version_summary = pd.DataFrame(version_rows, columns=["key", "value"])
save_csv(version_summary, "preprocessing", "dataset_version_summary.csv")
print(version_summary.to_string(index=False))


  STEP 16. Dataset version summary


  wrote results/preprocessing/dataset_version_summary.csv  (39 rows)
                             key                                                            value
                   timestamp_utc                                        2026-09-30T20:45:18+00:00
                      git_commit                         ec4cb4b853db5e018c02b7db76acbd86c54e6427
                            seed                                                               42
                    project_root                                             /content/BSM-Project
                      class_root                                /content/BSM-Project/data/raw/BSM
                     class_order          Ilmenite, Rutile, Zircon, Garnet, Monazite, Sillimanite
                       n_classes                                                                6
                      image_size                                                        224x224x3
            raw_files_discovered                 

---
## Notebook 01 complete

Generated artefacts:

| Path | Contents |
|---|---|
| `results/dataset_summary/raw_dataset_inventory.csv` | every file found on disk |
| `results/dataset_summary/dataset_summary.csv` | actual vs paper class counts |
| `results/dataset_summary/class_distribution.png` | class distribution |
| `results/dataset_summary/sample_images.png` | sample grid |
| `results/dataset_summary/image_dimensions.png` | dimension analysis |
| `results/preprocessing/data_wrangling_report.csv` | per-image keep/exclude + reason |
| `results/preprocessing/clean_dataset_manifest.csv` | retained images |
| `results/preprocessing/data_imputation_report.csv` | median imputation record |
| `results/preprocessing/final_split_manifest.csv` | every sample used, with group ids |
| `results/preprocessing/dataset_version_summary.csv` | dataset fingerprint |
| `results/augmentation/augmentation_manifest.csv` | per-variant augmentation record |
| `results/augmentation/augmentation_summary.csv` | augmentation counts |
| `results/augmentation/augmentation_count_comparison.csv` | paper vs actual, with reasons |
| `results/augmentation/augmentation_examples.png` | augmentation samples |

Next: **`02_Model_Experiments.ipynb`**.